In [6]:
import pandas as pd, requests, os

url = "https://dataverse.harvard.edu/api/access/datafile/4139573"
os.makedirs("../data", exist_ok=True)
path = "../data/drugbank_ddi.tab"

if not os.path.exists(path):
    r = requests.get(url)
    print("status:", r.status_code, "size:", len(r.content))
    open(path, "wb").write(r.content)

df = pd.read_csv(path, sep="\t")
print(df.shape)
print(df.columns.tolist())
df.head()

status: 200 size: 44381882
(191808, 6)
['ID1', 'ID2', 'Y', 'Map', 'X1', 'X2']


,ID1,ID2,Y,Map,X1,X2
0,DB04571,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC1=CC2=CC3=C(OC(=O)C=C3C)C(C)=C2O1,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
1,DB00855,DB00460,1,#Drug1 may increase the photosensitizing activ...,NCC(=O)CCC(O)=O,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
2,DB09536,DB00460,1,#Drug1 may increase the photosensitizing activ...,O=[Ti]=O,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
3,DB01600,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC(C(O)=O)C1=CC=C(S1)C(=O)C1=CC=CC=C1,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...
4,DB09000,DB00460,1,#Drug1 may increase the photosensitizing activ...,CC(CN(C)C)CN1C2=CC=CC=C2SC2=C1C=C(C=C2)C#N,COC(=O)CCC1=C2NC(\C=C3/N=C(/C=C4\N\C(=C/C5=N/C...


In [5]:
import os, re

os.remove("../data/drugbank_ddi.tab")  # delete the 404 junk

meta = open("../venv/Lib/site-packages/tdc/metadata.py", encoding="utf-8").read()
for line in meta.split("\n"):
    if "drugbank" in line.lower() or "twosides" in line.lower():
        print(line.strip())

ddi_dataset_names = ["drugbank", "twosides"]
"drugbank_drugs",
"drugbank": "tab",
"twosides": "csv",
"drugbank_drugs": "csv",
"drugbank": 4139573,
"twosides": 4139574,
"drugbank": 191808,
"twosides": 4649441,


In [7]:
drugs = set(df.ID1) | set(df.ID2)
print("unique drugs:", len(drugs))
print("interaction types:", df.Y.nunique())

counts = df.Y.value_counts()
print("\nmost common:")
print(counts.head(3))
print("\nrarest:")
print(counts.tail(3))

# what do the top types actually mean?
for y in counts.head(3).index:
    print(f"\nY={y} ({counts[y]} pairs): {df[df.Y==y].Map.iloc[0][:90]}")

unique drugs: 1706
interaction types: 86

most common:
Y
49    60751
47    34360
73    23779
Name: count, dtype: int64

rarest:
Y
52    10
26     7
42     6
Name: count, dtype: int64

Y=49 (60751 pairs): The risk or severity of adverse effects can be increased when #Drug1 is combined with #Dru

Y=47 (34360 pairs): The metabolism of #Drug2 can be decreased when combined with #Drug1.

Y=73 (23779 pairs): The serum concentration of #Drug2 can be increased when it is combined with #Drug1.
